# MultiLogiEval Lean: Error Classification Analysis

Comprehensive analysis of 26 verified-but-wrong cases using error root cause classification.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Define consistent color palette
ERROR_COLORS = {
    'AXIOMATIZES_CONCLUSION': '#e74c3c',
    'AXIOMATIZES_CONTRADICTION': '#3498db',
    'INCORRECT_FORMALIZATION': '#f39c12',
    'REASONING_FAILURE': '#2ecc71',
    'AXIOMATIZES_UNMENTIONED': '#9b59b6',
    'OTHER': '#95a5a6'
}

## 1. Load Data and Basic Statistics

In [ ]:
# Load the error analysis CSV
df = pd.read_csv('../results/multilogieval/all_depths/lean/error_root_cause_analysis.csv')

print(f"Total verified-but-wrong cases: {len(df)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData shape: {df.shape}")
print(f"\nUnique error types: {df['Root_Cause_Category'].nunique()}")
print(f"\nUnique patterns: {df['Pattern'].nunique()}")
print(f"\nLogic types: {df['Logic_Type'].unique()}")
print(f"\nDepth levels: {sorted(df['Depth'].unique())}")

In [ ]:
# Show first few rows
df.head()

## 2. Error Type Distribution

In [ ]:
# Calculate error distribution
error_dist = df['Root_Cause_Category'].value_counts()

print("Root Cause Distribution:")
print("=" * 70)
for category, count in error_dist.items():
    percentage = (count / len(df)) * 100
    print(f"{category:30s} {count:3d} cases ({percentage:5.1f}%)")
print(f"\nTotal: {len(df)} cases")

# Calculate axiomatization total
axiom_types = ['AXIOMATIZES_CONCLUSION', 'AXIOMATIZES_CONTRADICTION', 'AXIOMATIZES_UNMENTIONED']
axiom_total = sum(error_dist.get(t, 0) for t in axiom_types)
print(f"\nTotal Axiomatization Errors: {axiom_total} ({axiom_total/len(df)*100:.1f}%)")

In [ ]:
# Visualize error distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = [ERROR_COLORS.get(cat, '#95a5a6') for cat in error_dist.index]
error_dist.plot(kind='bar', ax=ax1, color=colors, edgecolor='black', linewidth=1.2)
ax1.set_title('Error Type Distribution (Bar Chart)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Error Category', fontsize=12)
ax1.set_ylabel('Number of Cases', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Add count labels on bars
for i, v in enumerate(error_dist.values):
    ax1.text(i, v + 0.3, str(v), ha='center', va='bottom', fontweight='bold')

# Pie chart
colors_pie = [ERROR_COLORS.get(cat, '#95a5a6') for cat in error_dist.index]
ax2.pie(error_dist.values, labels=error_dist.index, autopct='%1.1f%%', 
        colors=colors_pie, startangle=90, textprops={'fontsize': 10})
ax2.set_title('Error Type Distribution (Pie Chart)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Error Types by Prediction Pattern

In [ ]:
# Cross-tabulation of patterns and error types
pattern_error = pd.crosstab(df['Pattern'], df['Root_Cause_Category'])

print("Error Types by Pattern:")
print("=" * 70)
print(pattern_error)

print("\nKey insights:")
for pattern in pattern_error.index:
    top_error = pattern_error.loc[pattern].idxmax()
    count = pattern_error.loc[pattern].max()
    total = pattern_error.loc[pattern].sum()
    print(f"  {pattern} ({total} total): Most common is {top_error} ({count} cases)")

In [ ]:
# Visualize grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))

# Reorder columns by error type frequency
ordered_cols = error_dist.index.tolist()
pattern_error_ordered = pattern_error[ordered_cols]

pattern_error_ordered.plot(kind='bar', ax=ax, 
                           color=[ERROR_COLORS.get(col, '#95a5a6') for col in ordered_cols],
                           edgecolor='black', linewidth=1.2)
ax.set_title('Error Types by Prediction Pattern', fontsize=14, fontweight='bold')
ax.set_xlabel('Prediction Pattern (Ground Truth → Prediction)', fontsize=12)
ax.set_ylabel('Number of Cases', fontsize=12)
ax.legend(title='Error Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Error Distribution by Logic Type and Depth

In [ ]:
# Error distribution by logic type
logic_error = pd.crosstab(df['Logic_Type'], df['Root_Cause_Category'])

print("Error Types by Logic Type:")
print("=" * 70)
print(logic_error)

# Error distribution by depth
depth_error = pd.crosstab(df['Depth'], df['Root_Cause_Category'])

print("\nError Types by Depth:")
print("=" * 70)
print(depth_error)

In [ ]:
# Visualize logic type distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# By logic type
ordered_cols = error_dist.index.tolist()
logic_error_ordered = logic_error[ordered_cols]
logic_error_ordered.plot(kind='bar', ax=ax1,
                         color=[ERROR_COLORS.get(col, '#95a5a6') for col in ordered_cols],
                         edgecolor='black', linewidth=1.2)
ax1.set_title('Error Types by Logic Type', fontsize=14, fontweight='bold')
ax1.set_xlabel('Logic Type', fontsize=12)
ax1.set_ylabel('Number of Cases', fontsize=12)
ax1.legend(title='Error Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.tick_params(axis='x', rotation=0)
ax1.grid(axis='y', alpha=0.3)

# By depth
depth_error_ordered = depth_error[ordered_cols]
depth_error_ordered.plot(kind='bar', ax=ax2,
                         color=[ERROR_COLORS.get(col, '#95a5a6') for col in ordered_cols],
                         edgecolor='black', linewidth=1.2)
ax2.set_title('Error Types by Depth', fontsize=14, fontweight='bold')
ax2.set_xlabel('Depth Level', fontsize=12)
ax2.set_ylabel('Number of Cases', fontsize=12)
ax2.legend(title='Error Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.tick_params(axis='x', rotation=0)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Key Findings

In [ ]:
print("=" * 70)
print("KEY FINDINGS: MULTILOGIEVAL LEAN ERROR CLASSIFICATION")
print("=" * 70)

# Top 3 error types
top3 = error_dist.head(3)
print("\n1. TOP 3 ERROR TYPES:")
for i, (cat, count) in enumerate(top3.items(), 1):
    pct = count / len(df) * 100
    print(f"   #{i}: {cat}")
    print(f"       {count} cases ({pct:.1f}%)")

print(f"\n2. AXIOMATIZATION ERRORS:")
print(f"   Total: {axiom_total}/{len(df)} ({axiom_total/len(df)*100:.1f}%)")
if axiom_total > 0:
    print(f"   Model axiomatizes instead of proving in {axiom_total}/{len(df)} cases")

print(f"\n3. LOGIC TYPE BREAKDOWN:")
for logic in sorted(df['Logic_Type'].unique()):
    logic_count = len(df[df['Logic_Type'] == logic])
    if logic_count > 0:
        top_err = df[df['Logic_Type'] == logic]['Root_Cause_Category'].value_counts().iloc[0]
        top_err_name = df[df['Logic_Type'] == logic]['Root_Cause_Category'].value_counts().index[0]
        print(f"   {logic}: {logic_count} cases, most common: {top_err_name} ({top_err})")

print(f"\n4. DEPTH BREAKDOWN:")
for depth in sorted(df['Depth'].unique()):
    depth_count = len(df[df['Depth'] == depth])
    if depth_count > 0:
        top_err = df[df['Depth'] == depth]['Root_Cause_Category'].value_counts().iloc[0]
        top_err_name = df[df['Depth'] == depth]['Root_Cause_Category'].value_counts().index[0]
        print(f"   {depth}: {depth_count} cases, most common: {top_err_name} ({top_err})")

print(f"\n5. PATTERN ANALYSIS:")
for pattern in pattern_error.index:
    total = pattern_error.loc[pattern].sum()
    top_err = pattern_error.loc[pattern].idxmax()
    top_count = pattern_error.loc[pattern].max()
    print(f"   {pattern} ({total} cases):")
    print(f"     → {top_err}: {top_count} cases ({top_count/total*100:.1f}%)")

print("\n" + "=" * 70)
print("CONCLUSION:")
print("=" * 70)
print(f"\nMultiLogiEval shows {len(df)} verified-but-wrong cases across")
print(f"{df['Logic_Type'].nunique()} logic types and {df['Depth'].nunique()} depth levels.")
if axiom_total/len(df) > 0.5:
    print(f"\nAxiomatization errors dominate ({axiom_total/len(df)*100:.1f}%), indicating")
    print("Lean verification succeeds syntactically but proofs are logically invalid.")
print("=" * 70)

## 6. Example Cases from Top 3 Error Types

In [ ]:
# Get top 3 error types
top_3_types = error_dist.head(3).index.tolist()

for error_type in top_3_types:
    subset = df[df['Root_Cause_Category'] == error_type]
    count = len(subset)
    pct = count / len(df) * 100
    
    print("\n" + "=" * 70)
    print(f"{error_type} ({count} cases, {pct:.1f}%)")
    print("=" * 70)
    
    # Show up to 3 examples
    for idx, (_, row) in enumerate(subset.head(3).iterrows(), 1):
        print(f"\nExample {idx}:")
        print(f"  Logic: {row['Logic_Type']}, Depth: {row['Depth']}")
        print(f"  Pattern: {row['Pattern']}")
        print(f"  Context: {str(row['Context'])[:120]}...")
        print(f"  Question: {str(row['Question'])[:120]}...")
        if pd.notna(row['Problematic_Lines']):
            print(f"  Problematic Line: {row['Problematic_Lines']}")
        print(f"  Error: {str(row['Error_Description'])[:150]}...")